# Reduced OMEGA caustic diagnostic

This notebook isolates four primary ray tubes in a $2\times2$ lattice close to normal incidence for OMEGA beam 17. The narrow diagnostic spot places every primary at a small impact parameter, so its turning point should lie close to the spherical critical surface.

The plots follow each ray through both sheets and expose the quantities used by caustic detection: the signed projected ray-tube Jacobian and its non-negative magnitude, uncapped and capped amplitude, and their positive difference. The final section forms and plots both tetrahedral sheets.

In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

from pyGATH.fields import tetrahedralise_sheet_fields
from pyGATH.io import load_simulation_config
from pyGATH.plotting import plot_tetrahedral_mesh
from pyGATH.raytracing import (
    RAY_SHEET_LAYOUT,
    RAY_STATE_LAYOUT,
    critical_density,
)

plt.rcParams.update({"figure.dpi": 110})

In [ ]:
project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
config_path = (
    project_root / "configs" / "example_configs" / "omega_single_beam_caustic.toml"
)

started = time.perf_counter()
simulation = load_simulation_config(config_path)
grid = simulation.build_grid()
beams = simulation.load_beams()
initial_rays = simulation.initialize_rays(grid, beams)
jax.block_until_ready(initial_rays.state)
setup_seconds = time.perf_counter() - started

started = time.perf_counter()
trace = simulation.trace_rays(initial_rays, grid)
jax.block_until_ready(trace.sheet_fields)
trace_seconds = time.perf_counter() - started

print(f"JAX device: {jax.devices()[0]}")
print(f"Setup and ray initialization: {setup_seconds:.3f} s")
print(f"Trace and sheet construction: {trace_seconds:.3f} s")
print(f"Sheet shape: {trace.sheet_fields.shape}")
print(f"All primary rays exited: {bool(trace.terminated)}")
print(f"Caustics found: {int(np.asarray(trace.has_caustic).sum())} / 4")

## Expected and detected turning points

For a spherical density profile, the conserved angular momentum is $L=|\mathbf r\times\mathbf p|$. At the radial turning point, $\epsilon(r)=L^2/r^2$. The bisection below compares this prediction with the detected caustic radius.

In [ ]:
omega = float(beams.omega[0])
ncritical = float(critical_density(omega))
density_power = 3.78
critical_radius = 343.0e-6 * 1.165 ** (1.0 / density_power)


def analytic_permittivity(radius):
    return 1.0 - (critical_radius / radius) ** density_power


def predicted_turning_radius(angular_momentum):
    lower = critical_radius
    upper = float(grid.xb[-1])
    for _ in range(80):
        middle = 0.5 * (lower + upper)
        residual = analytic_permittivity(middle) - (angular_momentum / middle) ** 2
        if residual > 0.0:
            upper = middle
        else:
            lower = middle
    return 0.5 * (lower + upper)


initial_state = np.asarray(initial_rays.state[0])
impact_parameters = np.linalg.norm(
    initial_state[..., RAY_STATE_LAYOUT.impact_parameter], axis=-1
)
angular_momenta = np.linalg.norm(
    np.cross(
        initial_state[..., RAY_STATE_LAYOUT.position],
        initial_state[..., RAY_STATE_LAYOUT.momentum],
    ),
    axis=-1,
)
predicted_radii = np.vectorize(predicted_turning_radius)(angular_momenta)
caustic_positions = np.asarray(
    trace.sheet_fields[0, 0, ..., -1, RAY_STATE_LAYOUT.position]
)
detected_radii = np.linalg.norm(caustic_positions, axis=-1)
caustic_hydro = grid.interpolate(jnp.asarray(caustic_positions))
caustic_density_ratio = np.asarray(caustic_hydro.ne) / ncritical

print(f"Normal-incidence critical radius: {critical_radius * 1e6:.6f} um\n")
print(" ray    impact [um]   predicted r [um]   detected r [um]   ne/nc at detected")
for first_index in range(2):
    for second_index in range(2):
        label = f"({first_index}, {second_index})"
        print(
            f" {label:8s} {impact_parameters[first_index, second_index] * 1e6:10.4f}"
            f" {predicted_radii[first_index, second_index] * 1e6:18.6f}"
            f" {detected_radii[first_index, second_index] * 1e6:18.6f}"
            f" {caustic_density_ratio[first_index, second_index]:17.8f}"
        )

## Per-ray tube diagnostics

Sheet 1 and sheet 2 are joined at their common caustic sample. Solid area curves retain triangle orientation, while dotted curves show the absolute area used by the amplitude diagnostic. Markers denote the detected sheet boundary.

In [ ]:
sheet_fields = np.asarray(trace.sheet_fields[0])
ray_diagnostics = {}
number_of_samples = sheet_fields.shape[-2]

for first_index in range(2):
    for second_index in range(2):
        first_sheet = sheet_fields[0, first_index, second_index]
        second_sheet = sheet_fields[1, first_index, second_index]
        fields = np.concatenate((first_sheet, second_sheet[1:]), axis=0)
        positions = fields[:, RAY_STATE_LAYOUT.position]
        momenta = fields[:, RAY_STATE_LAYOUT.momentum]
        neighbours = fields[:, RAY_STATE_LAYOUT.neighbour_positions].reshape(-1, 3, 3)
        edge_one = neighbours[:, 1] - neighbours[:, 0]
        edge_two = neighbours[:, 2] - neighbours[:, 0]
        directions = momenta / np.linalg.norm(momenta, axis=-1, keepdims=True)
        signed_jacobian = 0.5 * np.einsum(
            "ij,ij->i", np.cross(edge_one, edge_two), directions
        )
        signed_jacobian_ratio = signed_jacobian / signed_jacobian[0]
        path = fields[:, RAY_STATE_LAYOUT.path_length]
        path = path - path[0]
        hydro = grid.interpolate(jnp.asarray(positions))
        ray_diagnostics[(first_index, second_index)] = {
            "fields": fields,
            "path": path,
            "radius": np.linalg.norm(positions, axis=-1),
            "density_ratio": np.asarray(hydro.ne) / ncritical,
            "signed_jacobian_ratio": signed_jacobian_ratio,
            "jacobian_magnitude_ratio": np.abs(signed_jacobian_ratio),
            "momentum_squared": np.sum(momenta**2, axis=-1),
            "caustic_index": number_of_samples - 1,
        }

In [ ]:
print("Ray event summary (path measured from grid entry)")
for ray_index, diagnostic in ray_diagnostics.items():
    path_um = diagnostic["path"] * 1.0e6
    radius_um = diagnostic["radius"] * 1.0e6
    signed_jacobian = diagnostic["signed_jacobian_ratio"]
    fields = diagnostic["fields"]
    score = (
        fields[:, RAY_SHEET_LAYOUT.uncapped_amplitude]
        - fields[:, RAY_SHEET_LAYOUT.capped_amplitude]
    )
    turning_index = int(np.argmin(diagnostic["radius"]))
    score_index = int(np.argmax(score))
    crossings = np.flatnonzero(signed_jacobian[:-1] * signed_jacobian[1:] <= 0.0)
    crossing_text = (
        ", ".join(
            f"{path_um[index]:.2f} um / r={radius_um[index]:.2f} um"
            for index in crossings
        )
        or "none on sheet samples"
    )
    print(f"\nray {ray_index}:")
    print(
        f"  minimum radius: {radius_um[turning_index]:.4f} um "
        f"at path {path_um[turning_index]:.2f} um"
    )
    print(
        f"  maximum score: {score[score_index]:.6g} at "
        f"r={radius_um[score_index]:.4f} um, path={path_um[score_index]:.2f} um"
    )
    print(f"  signed-Jacobian zero crossings: {crossing_text}")

In [ ]:
figure, axes = plt.subplots(
    3, 2, figsize=(14, 13), sharex=True, constrained_layout=True
)
colors = plt.get_cmap("tab10").colors

for color_index, (ray_index, diagnostic) in enumerate(ray_diagnostics.items()):
    color = colors[color_index]
    label = f"ray {ray_index}"
    path_um = diagnostic["path"] * 1.0e6
    caustic_index = diagnostic["caustic_index"]
    turning_index = int(np.argmin(diagnostic["radius"]))
    fields = diagnostic["fields"]
    uncapped = fields[:, RAY_SHEET_LAYOUT.uncapped_amplitude]
    capped = fields[:, RAY_SHEET_LAYOUT.capped_amplitude]
    permittivity = fields[:, RAY_STATE_LAYOUT.permittivity]

    plotted = (
        diagnostic["radius"] * 1.0e6,
        diagnostic["density_ratio"],
        diagnostic["signed_jacobian_ratio"],
        uncapped,
        uncapped - capped,
        permittivity,
    )
    for axis, values in zip(axes.flat, plotted, strict=True):
        axis.plot(path_um, values, color=color, label=label)
        axis.scatter(
            path_um[caustic_index], values[caustic_index], color=color, s=22, zorder=4
        )
        axis.scatter(
            path_um[turning_index],
            values[turning_index],
            facecolors="none",
            edgecolors=color,
            marker="s",
            s=30,
            zorder=4,
        )

    axes[1, 0].plot(
        path_um,
        diagnostic["jacobian_magnitude_ratio"],
        color=color,
        linestyle=":",
        alpha=0.8,
    )
    axes[1, 1].plot(path_um, capped, color=color, linestyle="--", alpha=0.9)
    axes[2, 1].plot(
        path_um, diagnostic["momentum_squared"], color=color, linestyle="--", alpha=0.9
    )

axes[0, 0].axhline(critical_radius * 1.0e6, color="black", linestyle="--")
axes[0, 0].set_ylabel("Radius [um]")
axes[0, 0].set_title("Radius (critical surface dashed)")
axes[0, 1].axhline(1.0, color="black", linestyle="--")
axes[0, 1].set_ylabel(r"$n_e/n_c$")
axes[0, 1].set_title("Electron density")
axes[1, 0].axhline(0.0, color="black", linewidth=0.8)
axes[1, 0].set_ylabel(r"$J/J_0$")
axes[1, 0].set_title("Signed projected Jacobian (solid), magnitude (dotted)")
axes[1, 1].set_yscale("log")
axes[1, 1].set_ylabel("Amplitude")
axes[1, 1].set_title("Uncapped (solid), capped (dashed)")
axes[2, 0].axhline(0.0, color="black", linewidth=0.8)
axes[2, 0].set_ylabel("Uncapped - capped amplitude")
axes[2, 0].set_title("Caustic score along sampled sheets")
axes[2, 1].set_ylabel(r"$\epsilon$ or $|p|^2$")
axes[2, 1].set_title("Permittivity (solid), momentum squared (dashed)")
for axis in axes[-1]:
    axis.set_xlabel("Path from grid entry [um]")
for axis in axes.flat:
    axis.grid(alpha=0.2)
axes[0, 0].legend(
    handles=[
        *(
            Line2D([0], [0], color=colors[index], label=f"ray {ray_index}")
            for index, ray_index in enumerate(ray_diagnostics)
        ),
        Line2D(
            [0],
            [0],
            color="black",
            marker="o",
            linestyle="none",
            label="detected split",
        ),
        Line2D(
            [0],
            [0],
            color="black",
            marker="s",
            markerfacecolor="none",
            linestyle="none",
            label="minimum radius",
        ),
    ],
    loc="best",
)
axes[1, 0].legend(
    handles=[
        Line2D([0], [0], color="black", label="signed Jacobian"),
        Line2D([0], [0], color="black", linestyle=":", label="magnitude"),
    ],
    loc="best",
)
axes[1, 1].legend(
    handles=[
        Line2D([0], [0], color="black", label="uncapped"),
        Line2D([0], [0], color="black", linestyle="--", label="capped"),
    ],
    loc="best",
)
figure.suptitle("Four near-normal OMEGA ray tubes", fontsize=15)

## Tetrahedral sheets

In [ ]:
started = time.perf_counter()
tetrahedral_field = tetrahedralise_sheet_fields(
    trace.sheet_fields,
    fields=("uncapped_amplitude", "capped_amplitude", "permittivity"),
)
jax.block_until_ready(tetrahedral_field.mesh.valid)
tetrahedral_seconds = time.perf_counter() - started
valid_counts = np.asarray(tetrahedral_field.mesh.valid[0]).sum(axis=1)
print(f"Tetrahedralisation: {tetrahedral_seconds:.3f} s")
print(f"Tetrahedra per sheet: {tetrahedral_field.mesh.ntetrahedra}")
print(f"Valid sheet 1 / sheet 2: {valid_counts.tolist()}")

In [ ]:
figure = plt.figure(figsize=(16, 7))
flat_caustics = caustic_positions.reshape(-1, 3)
for sheet_index in range(2):
    axis = figure.add_subplot(1, 2, sheet_index + 1, projection="3d")
    plot_tetrahedral_mesh(
        tetrahedral_field, beam_index=0, sheet_index=sheet_index, ax=axis
    )
    axis.scatter(
        *flat_caustics.T,
        color="red",
        marker="x",
        s=35,
        linewidth=1.0,
        label="Detected caustics",
    )
    axis.legend(loc="upper right")
    axis.view_init(elev=22, azim=-55)
    axis.set_title(f"Beam 17 near-normal tube, sheet {sheet_index + 1}")
figure.suptitle("Reduced four-ray tetrahedralisation")
figure.tight_layout()